# Back up the retained ForTy checkpoints

Select **CPU** under **Runtime > Change runtime type**. This notebook reads the
completed study from Google Drive, verifies every checkpoint against the
published retention manifest, and prepares **12 separate weight ZIPs plus one
metadata ZIP**. Each archive stays below 32 MiB. The recorded weight files total
about 294.6 MiB before archive metadata.

The backups preserve six best and six last paths, their model configurations
and receipts. Five best/last pairs have identical hashes; they remain separate
files. The study directory is read without modification.

There are three separate outcomes: source weights verified, runtime archives
prepared, and downloaded archives verified on your computer. Browser download
requests establish only that a transfer was requested. Complete the local
verification at the end before treating the backup as finished.

[Backup guide](https://github.com/CCC917111/geospatial-mamba-deforestation/blob/main/docs/checkpoint-backup.md)
· [Audited study](https://github.com/CCC917111/geospatial-mamba-deforestation/blob/main/results/architecture_pilot_t4_20261029/README.md)

The default source-upload mode works without cloning from GitHub. Upload the
supplied `checkpoint-backup-source.bundle` or the full project delivery ZIP
when prompted. The bundle is checked against a fixed hash before checkout.
Set `UPLOAD_SOURCE_BUNDLE=False` to clone the pinned source from GitHub.

In [ ]:
import datetime
import hashlib
import json
import os
from pathlib import Path
import re
import signal
import subprocess
import sys
import uuid

BACKUP_READY = False
BACKUP_SOURCE_COMMIT = "94da89fa4b8f1f42b458a2cd1f7a81b672f65aa4"
REPOSITORY = "https://github.com/CCC917111/geospatial-mamba-deforestation.git"
STUDY_ID = "forty-architecture-pilot-v1"
TRUSTED_MANIFEST_SHA256 = "66c668774ddba98f04bdc46a2ff3a5fd7fcd6ee07b208317b7dcd9d778460fb1"
UPLOAD_SOURCE_BUNDLE = True  # Upload the supplied bundle or project delivery ZIP.
TRUSTED_BUNDLE_SHA256 = "b29e63b0ec026399f3b9094b40cff63e4a27761bc49ca756755645fcc025e123"
TRUSTED_BUNDLE_BYTES = 1446268
DOWNLOAD_ALL_ARCHIVES = True  # Your browser may ask to allow multiple downloads.

if not re.fullmatch(r"[A-Za-z0-9_-]+", STUDY_ID):
    raise ValueError("Invalid study ID")
WORKSPACE = Path("/content/geospatial-checkpoint-backup")
WORKSPACE.mkdir(parents=True, exist_ok=True)
BACKUP_SOURCE_ROOT = WORKSPACE / ("source-" + uuid.uuid4().hex[:8])
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%M%SZ")
BACKUP_ROOT = WORKSPACE / ("backup-" + stamp + "-" + uuid.uuid4().hex[:8])
STUDY_ROOT = Path("/content/drive/MyDrive/geospatial-mamba/research") / STUDY_ID
print("Read-only study:", STUDY_ROOT)
print("Fresh runtime backup directory:", BACKUP_ROOT)

In [ ]:
import io
import stat
from pathlib import PurePosixPath
import zipfile


def source_bundle_bytes(filename, payload):
    if len(payload) >= 32 * 1024**2:
        raise ValueError("Source upload exceeds the 32 MiB budget")
    if filename.lower().endswith(".bundle"):
        content = payload
    elif filename.lower().endswith(".zip"):
        try:
            with zipfile.ZipFile(io.BytesIO(payload)) as archive:
                entries = archive.infolist()
                if len(entries) > 1000:
                    raise ValueError("Too many source ZIP entries")
                for entry in entries:
                    path = PurePosixPath(entry.filename)
                    if (path.is_absolute() or ".." in path.parts
                            or "\\" in entry.filename
                            or stat.S_ISLNK(entry.external_attr >> 16)):
                        raise ValueError("Unsafe source ZIP path")
                matches = [entry for entry in entries
                           if PurePosixPath(entry.filename).name == "checkpoint-backup-source.bundle"]
                if len(matches) != 1 or matches[0].file_size != TRUSTED_BUNDLE_BYTES:
                    raise ValueError("Select the package containing exactly one checkpoint-backup-source.bundle")
                content = archive.read(matches[0])
        except zipfile.BadZipFile as error:
            raise ValueError("Invalid source ZIP") from error
    else:
        raise ValueError("Upload checkpoint-backup-source.bundle or the project delivery ZIP")
    if (len(content) != TRUSTED_BUNDLE_BYTES
            or hashlib.sha256(content).hexdigest() != TRUSTED_BUNDLE_SHA256):
        raise ValueError("Source bundle does not match the trusted package")
    return content

In [ ]:
if UPLOAD_SOURCE_BUNDLE:
    from google.colab import files
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Select exactly one source bundle or project delivery ZIP")
    filename, payload = next(iter(uploaded.items()))
    content = source_bundle_bytes(filename, payload)
    bundle_path = WORKSPACE / ("source-" + uuid.uuid4().hex[:8] + ".bundle")
    with bundle_path.open("xb") as handle:
        handle.write(content)
    clone_source = str(bundle_path)
else:
    clone_source = REPOSITORY
subprocess.run(["git", "clone", "--no-checkout", clone_source, str(BACKUP_SOURCE_ROOT)], check=True)
subprocess.run(["git", "checkout", "--detach", BACKUP_SOURCE_COMMIT], cwd=BACKUP_SOURCE_ROOT, check=True)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=BACKUP_SOURCE_ROOT, text=True).strip()
dirty = subprocess.check_output(["git", "status", "--porcelain"], cwd=BACKUP_SOURCE_ROOT, text=True)
if head != BACKUP_SOURCE_COMMIT or dirty:
    raise RuntimeError("Pinned backup source mismatch or modified checkout")
MANIFEST = BACKUP_SOURCE_ROOT / "results/architecture_pilot_t4_20261029/metadata/checkpoint_manifest.json"
if hashlib.sha256(MANIFEST.read_bytes()).hexdigest() != TRUSTED_MANIFEST_SHA256:
    raise RuntimeError("Published retention manifest changed")
TOOL = BACKUP_SOURCE_ROOT / "scripts/checkpoint_backup.py"
print("Trusted retention manifest verified.")

In [ ]:
BACKUP_READY = False
from google.colab import drive

try:
    drive.mount("/content/drive", readonly=True)
except Exception as error:
    raise RuntimeError(
        "Complete Google Drive authorization: allow notebook credential access, "
        "select the account for this study and consent to all requested permissions "
        "(check every box). Then rerun this cell."
    ) from error
if not os.path.ismount("/content/drive") or not STUDY_ROOT.is_dir():
    raise RuntimeError("Mount Drive and check the study path before creating backups.")
print("Study found:", STUDY_ROOT)

In [ ]:
def execute_with_cleanup(argv, cwd, log, env=None):
    process = subprocess.Popen(argv, cwd=cwd, stdout=log,
                               stderr=subprocess.STDOUT, text=True,
                               start_new_session=True, env=env)
    try:
        return_code = process.wait()
    finally:
        # First allow the driver to record interruption and release its study lock.
        try:
            os.killpg(process.pid, signal.SIGINT)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            pass
        # Terminate the archive process and any child processes.
        try:
            os.killpg(process.pid, signal.SIGTERM)
        except ProcessLookupError:
            pass
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            pass
        finally:
            # Reap descendants even if the driver has already exited.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass
            process.wait()
    if return_code:
        raise subprocess.CalledProcessError(return_code, argv)

In [ ]:
BACKUP_READY = False
if not os.path.ismount("/content/drive") or not STUDY_ROOT.is_dir():
    raise RuntimeError("Complete the Drive mounting cell first.")
command = [sys.executable, str(TOOL), "create", "--run-root", str(STUDY_ROOT),
           "--manifest", str(MANIFEST), "--manifest-sha256", TRUSTED_MANIFEST_SHA256,
           "--output-dir", str(BACKUP_ROOT)]
log_path = WORKSPACE / ("backup-driver-" + uuid.uuid4().hex[:8] + ".log")
print("Checking weights and preparing archives; log:", log_path)
try:
    with log_path.open("x") as log:
        execute_with_cleanup(command, cwd=BACKUP_SOURCE_ROOT, log=log)
except (subprocess.CalledProcessError, KeyboardInterrupt):
    print(log_path.read_text()[-6000:])
    raise
index = json.loads((BACKUP_ROOT / "backup-index.json").read_text())
if (index["status"] != "archives_created_local_download_not_verified"
        or index["manifest_sha256"] != TRUSTED_MANIFEST_SHA256
        or index["checkpoint_count"] != 12
        or not index["source_checkpoint_bytes_verified"]
        or index["local_download_verified"]):
    raise RuntimeError("Unexpected backup completion receipt")
BACKUP_READY = True
print("Source checkpoint bytes verified; runtime archives prepared:", len(index["archives"]))
print("Local download verification remains pending.")
for number, entry in enumerate(index["archives"]):
    print(number, entry["filename"], entry["bytes"], "bytes")

In [ ]:
if not globals().get("BACKUP_READY", False):
    raise RuntimeError("Complete checkpoint verification and archive creation first.")
from google.colab import files

metadata_zip = BACKUP_ROOT / "backup-metadata.zip"
if (not metadata_zip.is_file() or metadata_zip.is_symlink()
        or metadata_zip.stat().st_size >= 32 * 1024**2):
    raise RuntimeError("Missing or oversized backup metadata archive")
files.download(str(metadata_zip))
if DOWNLOAD_ALL_ARCHIVES:
    for entry in index["archives"]:
        target = BACKUP_ROOT / entry["filename"]
        if (target.parent != BACKUP_ROOT or target.is_symlink()
                or target.stat().st_size != entry["bytes"]
                or target.stat().st_size >= 32 * 1024**2):
            raise RuntimeError("Invalid backup archive path or size")
        digest = hashlib.sha256()
        with target.open("rb") as handle:
            for block in iter(lambda: handle.read(1024 * 1024), b""):
                digest.update(block)
        if digest.hexdigest() != entry["sha256"]:
            raise RuntimeError("Archive changed before download")
        files.download(str(target))
    print("All 13 download requests sent. Confirm they arrived and run local verification below.")
else:
    print("Metadata ZIP requested. Use the numbered cell below for the twelve weight ZIPs.")

If your browser blocked multiple downloads, allow downloads for Colab or use
the following cell repeatedly with a different archive number from the list
above. Keep each archive's original filename and place all downloaded files
in one fresh folder. Runtime files under `/content` are temporary until saved
to your computer.

For a missed `backup-metadata.zip`, use Colab's Files panel to download it
from the printed backup directory. The numbered cell selects weight archives.

In [ ]:
ARCHIVE_NUMBER = None  # Set a number from the list to request one missed archive.
if not globals().get("BACKUP_READY", False):
    raise RuntimeError("Complete checkpoint verification and archive creation first.")
if ARCHIVE_NUMBER is None:
    print("For a missed download, set ARCHIVE_NUMBER to an entry from the archive list.")
else:
    if not isinstance(ARCHIVE_NUMBER, int) or not 0 <= ARCHIVE_NUMBER < len(index["archives"]):
        raise ValueError("Choose an archive number from the list")
    entry = index["archives"][ARCHIVE_NUMBER]
    target = BACKUP_ROOT / entry["filename"]
    if (target.parent != BACKUP_ROOT or target.is_symlink()
            or target.stat().st_size != entry["bytes"] or target.stat().st_size >= 32 * 1024**2):
        raise RuntimeError("Invalid backup archive")
    digest = hashlib.sha256()
    with target.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    if digest.hexdigest() != entry["sha256"]:
        raise RuntimeError("Archive changed before download")
    files.download(str(target))

## Verify the downloaded files on your computer

In the folder containing all 13 ZIPs, extract only `backup-metadata.zip`, then
run these commands with Python 3.10 or newer:

```bash
python -m zipfile -e backup-metadata.zip .
python verify_backup.py verify --backup-dir . --manifest checkpoint_manifest.json --manifest-sha256 66c668774ddba98f04bdc46a2ff3a5fd7fcd6ee07b208317b7dcd9d778460fb1 --report local-backup-verification.json
```

This reads every downloaded weight archive and checks sizes, hashes, model
configurations and checkpoint membership against the original trusted
manifest. Keep the successful JSON verification receipt with the backups.
The twelve weight ZIPs can remain compressed during verification.

Preserve the original metadata ZIP and the study's scientific source
`7f55b2c30c830693873e1fe1628c8e52208ed57d` for future restoration. These backup
files contain weights rather than optimizer state; restoring them is not
mid-training optimizer resumption.